In [ ]:
import numpy as np
import os
import subprocess
import pandas as pd
import random
import glob
from pathlib import Path

# =====================================================
# 1. Define the mall grid
# 0 = corridor, 1 = wall, 2 = shop, 3 = exit/opening
# =====================================================
grid = np.array([
    [1,1,1,3,1,3,1,3,1,3,1,3,1,3,1,1,1,1,1],
    [1,1,1,0,2,0,2,0,2,0,2,0,2,0,2,0,2,1,1],
    [1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1],
    [1,1,1,0,2,0,2,0,2,0,2,0,2,0,2,0,2,1,1],
    [1,1,2,0,2,0,2,0,2,0,2,0,2,0,2,0,2,1,1],
    [1,1,2,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1],
    [1,1,2,0,2,0,2,0,2,0,2,0,2,0,2,0,2,1,1],
    [1,1,2,0,2,0,2,0,2,0,2,0,2,0,2,0,2,1,1],
    [1,1,2,0,2,0,2,0,2,0,2,0,2,0,2,0,2,1,1],
    [1,1,2,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1],
    [1,1,1,0,2,0,2,0,2,0,2,0,2,0,2,0,1,1,1],
    [1,1,1,3,1,3,1,1,1,3,1,3,1,3,1,1,1,1,1]
])

rows, cols = grid.shape

# =====================================================
# 2. Create the simulation folder
# =====================================================
case_dir = Path("fds_cases")
case_dir.mkdir(exist_ok=True)

chid = "fire_model"
fds_file = case_dir / f"{chid}.fds"

# =====================================================
# 3. Remove old output files to avoid reading outdated results
# =====================================================
old_files = list(case_dir.glob(f"{chid}*"))

for old_file in old_files:
    if old_file.suffix.lower() != ".fds":
        try:
            old_file.unlink()
        except PermissionError:
            print(f"Could not delete old file: {old_file}")

# =====================================================
# 4. Select a random shop as the fire location
# =====================================================
shops = [(i, j) for i in range(rows) for j in range(cols) if grid[i, j] == 2]

if not shops:
    raise ValueError("No shops found in the grid.")

fire_shop = random.choice(shops)
i_fire, j_fire = fire_shop

print("Fire location:", fire_shop)

# =====================================================
# 5. Create the FDS input file
# =====================================================
with open(fds_file, "w") as f:
    f.write("&HEAD CHID='fire_model', TITLE='Mall Fire Simulation'/\n")
    f.write(f"&MESH IJK={cols},{rows},5 XB=0,{cols},0,{rows},0,3 /\n")
    f.write("&TIME T_END=60 /\n")
    f.write("&REAC FUEL='PROPANE' /\n")
    f.write("&SURF ID='FIRE', HRRPUA=500 /\n")

    # Add walls and openings
    for r in range(rows):
        for c in range(cols):
            y = rows - r

            if grid[r, c] == 1:
                f.write(f"&OBST XB={c},{c+1},{y-1},{y},0,3 /\n")

            elif grid[r, c] == 3:
                f.write(f"&OBST XB={c},{c+1},{y-1},{y},0,3 SURF_ID='OPEN' /\n")

    # Add the fire source inside the selected shop
    x_offset = random.uniform(0, 0.8)
    y_offset = random.uniform(0, 0.8)

    f.write(
        f"&OBST XB={j_fire + x_offset},{j_fire + 0.2 + x_offset},"
        f"{rows - i_fire - 1 + y_offset},{rows - i_fire - 0.2 + y_offset},"
        "0,0.5 SURF_ID='FIRE' /\n"
    )

    # Add temperature devices for all corridor cells
    for r in range(rows):
        for c in range(cols):
            if grid[r, c] == 0:
                x = c + 0.5
                y = rows - r - 0.5

                f.write(
                    f"&DEVC ID='Cell_{r}_{c}', "
                    f"QUANTITY='TEMPERATURE', "
                    f"XYZ={x},{y},0.5 /\n"
                )

    f.write("&TAIL /\n")

print("FDS file created")

# =====================================================
# 6. Run the FDS simulation
# =====================================================
print("Running FDS...")

fds_command_path = Path(r"FDS6\bin\fds_local.bat")

if fds_command_path.exists():
    fds_command = str(fds_command_path.resolve())
else:
    fds_command = r"FDS6\bin\fds_local.bat"

try:
    subprocess.run(
        [fds_command, fds_file.name],
        cwd=case_dir,
        check=True
    )
except subprocess.CalledProcessError as e:
    raise RuntimeError(
        "FDS simulation failed. Check the generated .out file for details."
    ) from e

print("Simulation finished")

# =====================================================
# 7. Search for the generated DEVC CSV file
# =====================================================
csv_files = list(case_dir.glob("*_devc.csv"))

if len(csv_files) == 0:
    available_files = [file.name for file in case_dir.glob("*")]

    print("No DEVC CSV file found.")
    print("Files found in simulation folder:")

    for file_name in available_files:
        print("-", file_name)

    raise FileNotFoundError(
        "No DEVC CSV file was generated. "
        "Check whether FDS ran correctly and whether DEVC outputs were created."
    )

csv_file = csv_files[0]
print("CSV found:", csv_file)

# =====================================================
# 8. Read the CSV file and convert it into a risk grid
# =====================================================
df = pd.read_csv(csv_file)

# Clean column names by removing quotes and extra spaces
df.columns = df.columns.str.replace('"', '', regex=False).str.strip()

time_steps = df.shape[0]
risk_grid = np.zeros((time_steps, rows, cols))

for r in range(rows):
    for c in range(cols):
        matches = [col for col in df.columns if col.startswith(f"Cell_{r}_{c}")]

        if matches:
            risk_grid[:, r, c] = df[matches[0]].values

# =====================================================
# 9. Create the training dataset
# =====================================================
training_data = []

for t in range(time_steps):
    for r in range(rows):
        for c in range(cols):
            if grid[r, c] == 0:
                training_data.append({
                    "fire_row": i_fire,
                    "fire_col": j_fire,
                    "time": t,
                    "cell_row": r,
                    "cell_col": c,
                    "risk": risk_grid[t, r, c]
                })

training_df = pd.DataFrame(training_data)

# =====================================================
# 10. Save the training dataset
# =====================================================
output_dataset = "fire_training_dataset.csv"
training_df.to_csv(output_dataset, index=False)

print("Dataset saved successfully:", output_dataset)
